# Imports

In [ ]:
import sklearn as sk
import numpy as np

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
np.set_printoptions(linewidth=100, precision=2)

In [ ]:
%load_ext autoreload
%autoreload 2

# Iris

In [ ]:
from utils.utils import load_iris, split_db_2to1

D,L = load_iris()
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

## Gaussian Models

### MVG

In [ ]:
from utils.utils import vcol, get_cov

u0 = vcol(DTR[:, LTR==0].mean(1))
c0 = get_cov(DTR[:, LTR==0])


u1 = vcol(DTR[:, LTR==1].mean(1))
c1 = get_cov(DTR[:, LTR==1])

u2 = vcol(DTR[:, LTR==2].mean(1))
c2 = get_cov(DTR[:, LTR==2])

In [ ]:
from utils.utils import loglikelihood, logpdf_GAU_ND

logpdf0 = logpdf_GAU_ND(DTE, u0, c0)
logpdf1 = logpdf_GAU_ND(DTE, u1, c1)
logpdf2 = logpdf_GAU_ND(DTE, u2, c2)

In [ ]:
logS = np.vstack((logpdf0, logpdf1, logpdf2))

In [ ]:
Pc = np.ones((3,1))/3

### Working Directly with densities

In [ ]:
S = np.exp(logS)

In [ ]:
SJoint = S*Pc

In [ ]:
SJoint_MVG = np.load('../data/SJoint_MVG.npy')
print(np.sum(SJoint-SJoint_MVG))

In [ ]:
from utils.utils import vrow

SMarginal = vrow(SJoint.sum(0))

In [ ]:
SPost = SJoint / SMarginal

In [ ]:
pred = np.argmax(SPost, axis=0)

In [ ]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

### Working in Log Domain

In [ ]:
logPc = np.log(Pc)

In [ ]:
logSJoint = logS + logPc

In [ ]:
from scipy.special import logsumexp

logSMarginal = vrow(logsumexp(logSJoint, axis=0))


In [ ]:
logSPost = logSJoint - logSMarginal
SPost = np.exp(logSPost)


In [ ]:
pred = np.argmax(SPost, axis=0)

In [ ]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

### Naive Bayes

In [ ]:
logpdf0 = logpdf_GAU_ND(DTE, u0, c0 * np.eye(c0.shape[0], c0.shape[1]))
logpdf1 = logpdf_GAU_ND(DTE, u1, c1 * np.eye(c1.shape[0], c1.shape[1]))
logpdf2 = logpdf_GAU_ND(DTE, u2, c2 * np.eye(c2.shape[0], c2.shape[1]))

In [ ]:
logS = np.vstack((logpdf0, logpdf1, logpdf2))
logSJoint = logS + logPc
logSMarginal = vrow(logsumexp(logSJoint, axis=0))
logSPost = logSJoint - logSMarginal
SPost = np.exp(logSPost)

In [ ]:
pred = np.argmax(SPost, axis=0)

In [ ]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

### Tied Variance

In [ ]:
from utils.utils import get_class_covariances

_, Sw = get_class_covariances(DTR, LTR, [0, 1, 2])

In [ ]:
logpdf0 = logpdf_GAU_ND(DTE, u0, Sw)
logpdf1 = logpdf_GAU_ND(DTE, u1, Sw)
logpdf2 = logpdf_GAU_ND(DTE, u2, Sw)

In [ ]:
logS = np.vstack((logpdf0, logpdf1, logpdf2))
logSJoint = logS + logPc
logSMarginal = vrow(logsumexp(logSJoint, axis=0))
logSPost = logSJoint - logSMarginal
SPost = np.exp(logSPost)

In [ ]:
pred = np.argmax(SPost, axis=0)

In [ ]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

## Binary Tasks

In [ ]:
D = D[:, L != 0]
L = L[L != 0]
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)
labels = [1, 2]

In [ ]:
u1 = vcol(DTR[:, LTR==1].mean(1))
c1 = get_cov(DTR[:, LTR==1])

u2 = vcol(DTR[:, LTR==2].mean(1))
c2 = get_cov(DTR[:, LTR==2])

In [ ]:
logpdf1 = logpdf_GAU_ND(DTE, u1, c1)
logpdf2 = logpdf_GAU_ND(DTE, u2, c2)

In [ ]:
llr = logpdf2 - logpdf1

In [ ]:
P1 = P2 = 0.5
t = 0

In [ ]:
pred = np.where(llr >= t, 2, 1)

In [ ]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

## Testing my own classifier Class

In [ ]:
from utils.utils import load_iris, split_db_2to1

D,L = load_iris()
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

### Multiclass

In [ ]:
from utils.BayesClassifier import MultiClassMVG, MultiClassNaiveBayes, MultiClassTiedVariance

mvg = MultiClassMVG()
nb = MultiClassNaiveBayes()
tv = MultiClassTiedVariance()

In [ ]:
Pc = np.ones((3,1))/3

In [ ]:
mvg.fit(DTR, LTR)
nb.fit(DTR, LTR)
tv.fit(DTR, LTR)

mvg.set_priors(Pc)
nb.set_priors(Pc)
tv.set_priors(Pc)

In [ ]:
mvg.predict(DTE)
nb.predict(DTE)
tv.predict(DTE)

In [ ]:
print("MVG: ", f"{mvg.evaluate(LTE):.4f}")
print("Naive Bayes: ", f"{nb.evaluate(LTE):.4f}")
print("Tied Variance: ", f"{tv.evaluate(LTE):.4f}")

### Binary

In [ ]:
D = D[:, L != 0]
L = L[L != 0]
L[L == 1] = 0
L[L == 2] = 1
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

In [ ]:
from utils.BayesClassifier import BinaryMVG

mvg = BinaryMVG()
Pc = np.ones((2,1))/2
mvg.set_priors(np.ones((2,1))/2)
mvg.set_threshold_via_prior_ratio()
mvg.fit(DTR, LTR)
mvg.predict(DTE)
print("MVG: ", f"{mvg.evaluate(LTE):.4f}")

# Project

Last lab, we fit a gaussian to each individual feature. It seems features 4 and 5 are not gaussian, so lets analyze the performance without them.

Besides, in 1st lab we saw that features 0|1 have similar mean and different variance, while 2|3 have similar variance but different mean. Lets analyze the performance using only a pair.

Finally, lets try applying PCS  to the data and see the results

| | MVG | Tied | NB |
| -----| --- | ---- | --- |
| All Features | 7% | 9.3% | 7.2% |
| 0, 1, 2, 3 | 7.4% | 9.4% | 7.5% |
| 0, 1 | 36% | 50% | 36% |
| 2, 3 | 8.8% | 9.5% | 8.8% |
| PCA | 8.9% | 9.3% | 9% |

## Data

In [ ]:
from utils.utils import load_data

PROJECT_FILE = "../data/trainData.txt"
D, L, labels = load_data(PROJECT_FILE)
feature_names = [f'F{i}' for i in range(0, D.shape[0])]
L = L.astype(int)
labels = [0, 1]
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

## Classifiers, Priors and Thresholds

In [ ]:
from utils.BayesClassifier import BinaryMVG, BinaryNaiveBayes, BinaryTiedVariance

mvg = BinaryMVG()
nb = BinaryNaiveBayes()
tv = BinaryTiedVariance()

In [ ]:
Pc = np.ones((2,1))/2
mvg.set_priors(Pc)
nb.set_priors(Pc)
tv.set_priors(Pc)

mvg.set_threshold_via_prior_ratio()
nb.set_threshold_via_prior_ratio()
tv.set_threshold_via_prior_ratio()

## All Features

In [ ]:
mvg.fit(DTR, LTR)
nb.fit(DTR, LTR)
tv.fit(DTR, LTR)

In [ ]:
mvg.predict(DTE)
nb.predict(DTE)
tv.predict(DTE)

In [ ]:
print("MVG: ", f"{mvg.evaluate(LTE):.4f}")
print("Naive Bayes: ", f"{nb.evaluate(LTE):.4f}")
print("Tied Variance: ", f"{tv.evaluate(LTE):.4f}")

### Investigating Feature Independence

Investigating the features, we found low correlation between features, making their independence assumption likely true. That allows the Naive Bayes Classifier to work properly

In [ ]:
c0 = get_cov(DTR[:, LTR==0])
c1 = get_cov(DTR[:, LTR==1])

In [ ]:
print(c0, "\n\n", c1)

In [ ]:
Corr0 = c0 / ( vcol(c0.diagonal()**0.5) * vrow(c0.diagonal()**0.5) )
Corr1 = c1 / ( vcol(c1.diagonal()**0.5) * vrow(c1.diagonal()**0.5) )

print(Corr0, "\n\n", Corr1)

## Selecting Features

In [ ]:
D, L, labels = load_data(PROJECT_FILE)
D = D[2:5, :]
L = L.astype(int)

(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

In [ ]:
mvg.fit(DTR, LTR)
nb.fit(DTR, LTR)
tv.fit(DTR, LTR)

In [ ]:
mvg.predict(DTE)
nb.predict(DTE)
tv.predict(DTE)

In [ ]:
print("MVG: ", f"{mvg.evaluate(LTE):.4f}")
print("Naive Bayes: ", f"{nb.evaluate(LTE):.4f}")
print("Tied Variance: ", f"{tv.evaluate(LTE):.4f}")

## Including PCA

In [ ]:
from utils.utils import get_PCs

D, L, labels = load_data(PROJECT_FILE)
L = L.astype(int)
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

P = get_PCs(DTR, 3)
DTR_PCA = P.T @ DTR
DTE_PCA = P.T @ DTE

In [ ]:
mvg.fit(DTR_PCA, LTR)
nb.fit(DTR_PCA, LTR)
tv.fit(DTR_PCA, LTR)

In [ ]:
mvg.predict(DTE_PCA)
nb.predict(DTE_PCA)
tv.predict(DTE_PCA)

In [ ]:

print("MVG: ", f"{mvg.evaluate(LTE):.4f}")
print("Naive Bayes: ", f"{nb.evaluate(LTE):.4f}")
print("Tied Variance: ", f"{tv.evaluate(LTE):.4f}")